In [ ]:
# Install required packages

!pip install -q -U langgraph langchain-core langchain-openai openai nest_asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
#        LANGGRAPH AGENT USING OPENROUTER
#        EXPENSE TRACKER PROJECT
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import csv
import asyncio
import nest_asyncio

from google.colab import userdata

from langchain_core.tools import StructuredTool

from langchain.agents import create_agent

from langchain_openai import ChatOpenAI


# Apply nest_asyncio for Google Colab
nest_asyncio.apply()


# ============================================================
# 2. OPENROUTER API & MODEL SETUP
# ============================================================

# Get OpenRouter API key from Colab Secrets

OPENROUTER_API_KEY = userdata.get(
    "GenAi_Chatbot"
)


# Check API key

if not OPENROUTER_API_KEY:

    raise ValueError(
        "OPENROUTER_API_KEY not found. "
        "Please add it to Colab Secrets."
    )


print(
    "OpenRouter API key loaded successfully."
)


# ============================================================
# 3. INITIALIZE OPENROUTER LLM
# ============================================================

llm = ChatOpenAI(

    # OpenRouter API URL
    base_url="https://openrouter.ai/api/v1",

    # API key
    api_key=OPENROUTER_API_KEY,

    # OpenRouter model
    model="openai/gpt-oss-20b",

    # Consistent responses
    temperature=0
)


print(
    "OpenRouter LLM initialized successfully."
)


# ============================================================
# 4. EXPENSE FILE
# ============================================================

CSV_FILE = "expenses.csv"


# ============================================================
# 5. INITIALIZE CSV
# ============================================================

def _initialize_csv():

    # Check whether CSV exists

    if not os.path.exists(CSV_FILE):

        # Create CSV file

        with open(
            CSV_FILE,
            mode="w",
            newline=""
        ) as f:

            writer = csv.writer(f)

            # CSV headings

            writer.writerow([
                "Item",
                "Amount",
                "Category"
            ])


# ============================================================
# 6. ADD EXPENSE TOOL
# ============================================================

def add_expense(
    item: str,
    amount: float,
    category: str
) -> str:

    """
    Logs a new expense with item,
    amount and category.
    """

    # Initialize CSV
    _initialize_csv()


    # Open CSV in append mode

    with open(
        CSV_FILE,
        mode="a",
        newline=""
    ) as f:

        writer = csv.writer(f)

        # Store expense

        writer.writerow([
            item,
            amount,
            category
        ])


    # Return confirmation

    return (
        f"Logged: {item} - "
        f"${amount} ({category})"
    )


# ============================================================
# 7. GET EXPENSES TOOL
# ============================================================

def get_expenses() -> str:

    """
    Retrieves all logged expenses
    from the CSV file.
    """

    # Initialize CSV

    _initialize_csv()


    # Open CSV

    with open(
        CSV_FILE,
        mode="r"
    ) as f:

        rows = list(
            csv.reader(f)
        )


    # Check whether expenses exist

    if len(rows) <= 1:

        return "No expenses."


    # Return expenses

    return "\n".join(

        [
            ", ".join(row)
            for row in rows[1:]
        ]

    )


# ============================================================
# 8. WRAP TOOLS
# ============================================================

mcp_tools = [

    # Add expense tool

    StructuredTool.from_function(

        func=add_expense,

        name="add_expense",

        description=(
            "Logs a new expense "
            "with item, amount and category."
        )

    ),


    # Get expenses tool

    StructuredTool.from_function(

        func=get_expenses,

        name="get_expenses",

        description=(
            "Retrieves all logged "
            "expenses from the expense tracker."
        )

    )

]


print(
    "Expense tools created successfully."
)


# ============================================================
# 9. BUILD LANGGRAPH AGENT
# ============================================================

agent = create_agent(
    model=llm,
    tools=mcp_tools,
    system_prompt="""
You are an expense tracking assistant.

When an expense is added, give a short confirmation.
When expenses are retrieved, display the records clearly.

Do not add unnecessary explanations.
Keep responses short and simple.
"""
)


# ============================================================
# 10. HELPER FUNCTION
# ============================================================

def get_clean_text(content) -> str:

    # If response is string

    if isinstance(content, str):

        return content


    # If response is a list

    if isinstance(content, list):

        text_parts = []

        for block in content:

            if (
                isinstance(block, dict)
                and block.get("type") == "text"
            ):

                text_parts.append(
                    block.get("text", "")
                )

        return "".join(text_parts)


    # Convert anything else to string

    return str(content)


# ============================================================
# 11. AGENTIC WORKFLOW
# ============================================================

async def run_agentic_workflow():


    # --------------------------------------------------------
    # TASK 1: LOG EXPENSE
    # --------------------------------------------------------

    print(
        "--- Task 1: Log an expense ---"
    )


    prompt_1 = (
        "I bought a pizza for $12.50. "
        "Category is Food."
    )


    # Call LangGraph agent

    response_1 = await agent.ainvoke({

        "messages": [

            (
                "user",
                prompt_1
            )

        ]

    })


    # Get final message

    final_message_1 = response_1[
        "messages"
    ][-1]


    # Print response

    print(
        "\n[Agent Response]:"
    )

    print(
        get_clean_text(
            final_message_1.content
        )
    )


    # --------------------------------------------------------
    # TASK 2: RETRIEVE EXPENSES
    # --------------------------------------------------------

    print(
        "\n--- Task 2: Retrieve records ---"
    )


    prompt_2 = (
        "Show me all expenses logged so far."
    )


    # Call LangGraph agent

    response_2 = await agent.ainvoke({

        "messages": [

            (
                "user",
                prompt_2
            )

        ]

    })


    # Get final message

    final_message_2 = response_2[
        "messages"
    ][-1]


    # Print response

    print(
        "\n[Agent Response]:"
    )

    print(
        get_clean_text(
            final_message_2.content
        )
    )


# ============================================================
# 12. RUN WORKFLOW
# ============================================================

asyncio.run(
    run_agentic_workflow()
)

OpenRouter API key loaded successfully.
OpenRouter LLM initialized successfully.
Expense tools created successfully.
--- Task 1: Log an expense ---

[Agent Response]:
Logged: pizza - $12.5 (Food)

--- Task 2: Retrieve records ---

[Agent Response]:
- pizza: $12.5 (Food)  
- pizza: $12.5 (Food)
